# Recuperación sobre la normativa de la secretaría

Este cuaderno monta un sistema de recuperación completo sobre el corpus de la secretaría académica y, sobre todo, lo **mide**. La pregunta que contesta no es "¿funciona un RAG?" sino "¿cuánto mejor funciona con estas decisiones que con estas otras?".

Se apoya en tres capítulos del manual: [ingeniería de contexto](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/intro.html), [recuperación](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/rag.html) e [inferencia](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/inferencia.html).

Todo se ejecuta en local. No hace falta clave de ninguna API: los embeddings salen de un modelo pequeño de `sentence-transformers`, el índice es [LanceDB](https://lancedb.com/) sobre ficheros y el generador es `Qwen3-0.6B`. Nada de esto sale de vuestra máquina.

## Lo que vamos a ver

1. Trocear el corpus de dos maneras distintas.
2. Convertir texto en vectores y medir similitud a mano, con tres métricas.
3. Guardar los vectores en LanceDB y buscar.
4. Añadir búsqueda léxica y fundirla con la vectorial.
5. **Medir las tres búsquedas sobre diez preguntas reales.**
6. Meter lo recuperado en un modelo y comparar la respuesta con y sin contexto.

El paso 5 es el que importa. Los otros cuatro son fáciles de encontrar en cualquier tutorial; el que casi nunca aparece es el de comprobar si lo que has montado sirve para algo.

## Preparación

La primera celda instala las dependencias. En Colab tarda un par de minutos.

In [ ]:
!pip install -q lancedb tantivy duckdb sentence-transformers "transformers>=4.51" torch

La segunda localiza el corpus y el almacén de la secretaría. En local ya están; en Colab se clona el repositorio del manual.

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()

## El corpus

Cinco documentos en markdown: la normativa de matrícula, el calendario académico y tres guías docentes. Están escritos en registro administrativo, que es un registro distinto del que usa un alumno cuando pregunta.

Esa distancia es justo la dificultad del problema. Nadie escribe "¿cuál es el plazo de presentación de la solicitud de beca general?". Lo que se escribe es "¿hasta cuándo puedo pedir la beca?".

In [ ]:
docs = ctx.documentos()

for d in docs:
    print(f"{d['id']:24s} {len(d['texto']):6d} caracteres   {d['metadatos'].get('titulo', '')}")

print(f"\nTotal: {sum(len(d['texto']) for d in docs)} caracteres")

In [ ]:
# Un vistazo al documento más importante
print(docs[-1]["texto"][:900])

## Trocear

Un documento entero no cabe en el contexto, y aunque cupiera no interesa: si le damos al modelo veinte páginas para que responda sobre un plazo, estamos pagando veinte páginas y además diluyendo la respuesta entre ruido.

Hay que partirlo. La pregunta es por dónde, y las dos respuestas habituales son muy distintas:

* **Por tamaño fijo.** Cortar cada 600 caracteres con algo de solapa. No sabe nada del documento, así que parte frases y tablas por la mitad, pero funciona con cualquier cosa.
* **Por estructura.** Cortar por los encabezados. Cada trozo es un artículo completo, con su título. Requiere que el documento tenga estructura, que aquí la tiene.

La intuición dice que la segunda es mejor. Vamos a comprobarlo en lugar de creerlo.

In [ ]:
import re

import numpy as np


def trocear_fijo(texto, tam=600, solapa=100):
    """Corta cada `tam` caracteres, repitiendo `solapa` para no partir ideas."""
    trozos, i = [], 0
    while i < len(texto):
        trozos.append(texto[i:i + tam])
        i += tam - solapa
    return [t for t in trozos if len(t.strip()) > 80]


def trocear_por_seccion(texto):
    """Corta por encabezados de markdown de nivel 2 y 3.

    El `(?=...)` es una anticipación: parte justo antes del encabezado sin
    consumirlo, de modo que cada trozo empieza por su propio título.
    """
    partes = re.split(r"\n(?=#{2,3} )", texto)
    return [p.strip() for p in partes if len(p.strip()) > 80]


for nombre, trocear in [("fijo", trocear_fijo), ("por sección", trocear_por_seccion)]:
    trozos = [t for d in docs for t in trocear(d["texto"])]
    largos = [len(t) for t in trozos]
    print(f"{nombre:12s} {len(trozos):3d} trozos | "
          f"media {np.mean(largos):5.0f} | min {min(largos):3d} | max {max(largos):3d}")

Fijaos en la dispersión. El troceado fijo produce trozos casi todos del mismo tamaño, que es cómodo para el modelo de embeddings. El estructural produce trozos que van de poco más de cien caracteres a casi ochocientos, porque los artículos de una normativa no miden todos lo mismo.

Eso tiene una consecuencia que se ve más adelante: un trozo muy corto y uno muy largo no compiten en igualdad de condiciones cuando se comparan vectores.

## Del texto al vector

Un modelo de embeddings convierte un texto en un punto de un espacio de varios cientos de dimensiones, colocado de forma que los textos que hablan de lo mismo caigan cerca.

Usamos un modelo multilingüe pequeño. Es importante que sea multilingüe: la mayoría de los modelos de embeddings populares están entrenados sobre todo en inglés y rinden bastante peor en español, que es exactamente nuestro caso.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer(ctx.embeddings)

print("Modelo:", ctx.embeddings)
print("Dimensiones:", modelo.get_embedding_dimension())

v = modelo.encode("¿Hasta cuándo puedo pedir la beca?")
print("\nPrimeros 8 números del vector:")
print(np.round(v[:8], 4))
print("\nNorma del vector:", round(float(np.linalg.norm(v)), 3))

Ese vector no dice nada por sí solo, y es normal. Lo que tiene sentido es la **distancia entre dos vectores**, no el valor de sus componentes.

Fijaos en la norma. No vale uno. Esto va a importar en la sección siguiente más de lo que parece.

## Tres formas de medir el parecido

Hay tres métricas que se usan casi siempre, y conviene entender en qué se diferencian porque elegir mal degrada la recuperación en silencio.

* **Producto escalar**: $q \cdot d$. Rápido, pero crece con la longitud de los vectores.
* **Similitud del coseno**: $\frac{q \cdot d}{\|q\| \|d\|}$. El coseno del ángulo. Ignora la longitud y mira solo la dirección.
* **Distancia euclídea**: $\|q - d\|$. La distancia de toda la vida. Aquí la negamos para que, como en las otras dos, más grande signifique más parecido.

Vamos a medir la misma pregunta contra tres frases: una que responde, una del mismo dominio que no responde, y una que no tiene nada que ver.

In [ ]:
pregunta = "¿Hasta cuándo puedo pedir la beca general?"

frases = [
    "La solicitud de beca general se presenta del 1 de agosto al 15 de octubre.",
    "El plazo de matrícula ordinaria va del 15 al 31 de julio.",
    "Los gatos duermen una media de dieciséis horas al día.",
]

V = modelo.encode([pregunta] + frases)
q, D = V[0], V[1:]


def coseno(q, D):
    return (D @ q) / (np.linalg.norm(D, axis=1) * np.linalg.norm(q))


def escalar(q, D):
    return D @ q


def euclidea(q, D):
    return -np.linalg.norm(D - q, axis=1)


print("Normas de las tres frases:", np.round(np.linalg.norm(D, axis=1), 2))
print()

for nombre, metrica in [("coseno", coseno), ("escalar", escalar), ("euclídea", euclidea)]:
    s = metrica(q, D)
    print(f"{nombre:9s} {np.round(s, 3)}   ranking: {np.argsort(-s).tolist()}")

Aquí está el detalle que casi nunca se cuenta.

**El coseno y el producto escalar dan rankings distintos.** El coseno pone primero la frase de la beca, que es la correcta. El producto escalar pone primero la de la matrícula, que es incorrecta.

La razón está en la primera línea: las tres frases tienen normas distintas. El producto escalar premia a los vectores largos por el mero hecho de serlo, y la frase de la matrícula tiene una norma mayor. Un texto más largo o con más contenido tiende a producir un vector más largo, así que usar producto escalar sobre vectores sin normalizar equivale a **favorecer sistemáticamente a los trozos largos**.

La solución es normalizar los vectores a longitud uno. Con vectores normalizados las tres métricas ordenan igual, y eso conviene comprobarlo en lugar de fiarse.

In [ ]:
Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
qn = q / np.linalg.norm(q)

print("Normas tras normalizar:", np.round(np.linalg.norm(Dn, axis=1), 2))
print()

for nombre, metrica in [("coseno", coseno), ("escalar", escalar), ("euclídea", euclidea)]:
    s = metrica(qn, Dn)
    print(f"{nombre:9s} {np.round(s, 3)}   ranking: {np.argsort(-s).tolist()}")

Las tres coinciden. Y no es casualidad: con vectores normalizados, el producto escalar **es** el coseno, y la distancia euclídea es una función decreciente del coseno, así que el orden que producen es necesariamente el mismo.

De aquí sale una regla práctica que ahorra muchos disgustos:

> Normalizad los vectores al guardarlos. Así la métrica que elija el índice deja de importar, y os quitáis de encima una clase entera de fallos silenciosos.

Es lo que vamos a hacer a partir de ahora.

## Guardar los vectores: LanceDB

Calcular la similitud contra todos los trozos con `numpy` funciona con cincuenta trozos y deja de funcionar con cincuenta mil. Para eso están las bases de datos vectoriales.

Usamos [LanceDB](https://lancedb.com/) por una razón muy concreta: **es un fichero**. No hay servidor que levantar, ni contenedor, ni cuenta que crear, ni clave que pedir. Se comporta como SQLite pero para vectores, y eso la hace ideal para aprender y para prototipar. Cuando el proyecto crezca habrá que mirar otras opciones, y esa discusión está en el [capítulo de recuperación](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/rag.html).

Montamos la tabla con el troceado estructural y los vectores ya normalizados.

In [ ]:
import lancedb
from lancedb.index import FTS


def construir_tabla(db, nombre, trocear):
    """Trocea el corpus, calcula los vectores y los guarda en LanceDB."""
    filas = []
    for d in docs:
        titulo = d["metadatos"].get("titulo", d["id"])
        for i, trozo in enumerate(trocear(d["texto"])):
            filas.append({
                "id": f"{d['id']}#{i}",
                "documento": d["id"],
                "titulo": titulo,
                "texto": trozo,
            })

    vectores = modelo.encode([f["texto"] for f in filas], show_progress_bar=False)
    vectores = vectores / np.linalg.norm(vectores, axis=1, keepdims=True)
    for fila, vector in zip(filas, vectores):
        fila["vector"] = vector.astype("float32")

    tabla = db.create_table(nombre, data=filas, mode="overwrite")
    tabla.create_index("texto", config=FTS(), replace=True)  # para la búsqueda léxica
    return tabla


db = lancedb.connect("lancedb")
tabla = construir_tabla(db, "seccion", trocear_por_seccion)

print(f"{tabla.count_rows()} trozos indexados")

### Búsqueda vectorial

Se codifica la pregunta con el mismo modelo, se normaliza igual que los documentos y se pide a LanceDB los trozos más cercanos.

In [ ]:
def buscar_densa(tabla, pregunta, k=3):
    v = modelo.encode(pregunta)
    v = v / np.linalg.norm(v)
    return tabla.search(v.astype("float32")).metric("l2").limit(k).to_list()


def mostrar(resultados):
    for i, r in enumerate(resultados, 1):
        primera_linea = r["texto"].split("\n")[0][:70]
        print(f"  {i}. {r['id']:28s} {primera_linea}")


print("¿hasta cuándo puedo pedir la beca?")
mostrar(buscar_densa(tabla, "¿hasta cuándo puedo pedir la beca?"))

print("\n¿qué dice el artículo 21?")
mostrar(buscar_densa(tabla, "¿qué dice el artículo 21?"))

Vaya.

La primera pregunta devuelve trozos del calendario que hablan de fechas y de periodos, pero no el artículo 18, que es donde está la respuesta. La segunda es peor, y es casi cómica: se le pide el artículo 21 y devuelve el **artículo 1**.

(De paso habréis visto aparecer un par de trozos que son solo `---`. Es el frontmatter de los ficheros, que se ha indexado como si fuera contenido. Es un fallo real y muy común: la mitad del trabajo de montar un RAG consiste en limpiar lo que no debería estar en el índice.)

Esto no es un fallo de la instalación ni un error nuestro. Es **exactamente lo que hace un modelo de embeddings**: mide parecido semántico. "Artículo 21" y "artículo 18" se parecen muchísimo en significado, porque la diferencia entre ellos es un número, y los números es lo que peor representan estos modelos.

Conviene decirlo sin rodeos, porque es lo que más se malinterpreta de esta tecnología: **la búsqueda vectorial es mala buscando términos exactos**. Códigos, referencias, nombres propios y números son su punto débil, y da la casualidad de que una secretaría académica está llena de códigos, referencias y números.

### Búsqueda léxica

Para eso sigue existiendo la búsqueda de toda la vida, la que cuenta palabras. LanceDB la trae incorporada mediante un índice de texto completo, que es lo que ya hemos creado con `create_index(..., config=FTS())`.

Es la misma familia de técnicas que se ve en el [capítulo de fundamentos](https://iraitzm.github.io/manual-ia-generativa/parts/fundamentos/nlp.html): pesar cada palabra por lo rara que es, y premiar los documentos que contienen las palabras raras de la consulta.

In [ ]:
def buscar_lexica(tabla, pregunta, k=3):
    return tabla.search(pregunta, query_type="fts").limit(k).to_list()


print("¿qué dice el artículo 21?")
mostrar(buscar_lexica(tabla, "¿qué dice el artículo 21?"))

print("\n¿hasta cuándo puedo pedir la beca general?")
mostrar(buscar_lexica(tabla, "¿hasta cuándo puedo pedir la beca general?"))

Las dos preguntas que la búsqueda vectorial fallaba, la léxica las acierta.

Pero antes de declararla ganadora, probemos con una pregunta donde el alumno no use ninguna de las palabras del documento.

In [ ]:
# La normativa habla de "actas" y de "quince días hábiles".
# El alumno habla de "nota" y de "publicar". Ninguna palabra coincide.
pregunta = "no me ha llegado la nota, ¿cuándo la publican?"

print("LÉXICA:")
mostrar(buscar_lexica(tabla, pregunta))

print("\nVECTORIAL:")
mostrar(buscar_densa(tabla, pregunta))

Y aquí se da la vuelta la tortilla. La léxica no encuentra nada útil porque no hay palabras compartidas; la vectorial sí, porque entiende que "nota" y "acta" están relacionados.

O sea que ninguna de las dos gana siempre:

| | Vectorial | Léxica |
|---|---|---|
| Términos exactos, códigos, números | Mal | Bien |
| Sinónimos y paráfrasis | Bien | Mal |
| Idioma distinto del documento | Bien | Muy mal |
| Términos que no vio al entrenarse | Mal | Bien |

La respuesta no es elegir. Es usar las dos.

## Búsqueda híbrida

Fundir dos rankings tiene una trampa: sus puntuaciones no son comparables. La distancia vectorial y la puntuación del índice de texto viven en escalas distintas y no se pueden sumar.

La solución habitual se llama **fusión recíproca de rangos** (RRF), y su gracia es que tira las puntuaciones a la basura y se queda solo con la **posición**. Cada resultado suma $\frac{1}{k + \text{posición}}$ por cada lista en la que aparece.

Son cinco líneas, y merece la pena escribirlas a mano en lugar de invocar la función de un framework, porque así se ve que no hay magia.

La constante $k$ (60 por convención) amortigua las primeras posiciones: sin ella, el primer puesto valdría el doble que el segundo, lo cual es demasiado dramático. Su efecto real es que **un resultado que sale razonablemente bien en las dos listas gana a uno que sale primero en una sola**.

In [ ]:
def rrf(listas, k=60):
    """Fusiona varias listas de identificadores por su posición."""
    puntos = {}
    for lista in listas:
        for posicion, ident in enumerate(lista, start=1):
            puntos[ident] = puntos.get(ident, 0) + 1 / (k + posicion)
    return sorted(puntos, key=puntos.get, reverse=True)


def buscar_hibrida(tabla, pregunta, k=3):
    densa = [r["id"] for r in buscar_densa(tabla, pregunta, k=5)]
    lexica = [r["id"] for r in buscar_lexica(tabla, pregunta, k=5)]
    return rrf([densa, lexica])[:k]


for pregunta in ["¿qué dice el artículo 21?",
                 "no me ha llegado la nota, ¿cuándo la publican?"]:
    print(pregunta)
    print(f"  densa   : {[r['id'] for r in buscar_densa(tabla, pregunta)]}")
    print(f"  léxica  : {[r['id'] for r in buscar_lexica(tabla, pregunta)]}")
    print(f"  híbrida : {buscar_hibrida(tabla, pregunta)}")
    print()

## Medir

Todo lo anterior son anécdotas. Dos preguntas bien elegidas demuestran lo que uno quiera demostrar, y por eso los tutoriales de RAG siempre funcionan.

Vamos a hacer lo que hay que hacer: un conjunto de preguntas escritas como las escribiría un alumno, cada una con el fragmento de texto que **tiene** que aparecer en lo recuperado para dar la respuesta por buena.

La métrica es **recall@3**: de diez preguntas, ¿en cuántas aparece el fragmento correcto entre los tres primeros resultados? Se elige 3 porque es lo que de verdad le vamos a pasar al modelo.

In [ ]:
# (pregunta tal y como la escribiría un alumno, texto que debe aparecer)
CASOS = [
    ("¿hasta cuándo puedo pedir la beca general?", "1 de agosto al 15 de octubre"),
    ("¿qué dice el artículo 21?", "Artículo 21"),
    ("¿puedo defender el TFG si me queda una asignatura?", "no se admite a defensa"),
    ("no me ha llegado la nota, ¿cuándo la publican?", "quince días hábiles"),
    ("quiero quitarme una asignatura que cogí en julio", "alta o la baja de asignaturas"),
    ("¿cuántas veces me puedo presentar a un examen?", "seis convocatorias"),
    ("me han suspendido y no estoy de acuerdo", "cinco días hábiles"),
    ("¿cuántos créditos tengo que aprobar el primer año?", "doce créditos"),
    ("trabajo por las mañanas, ¿puedo cambiarme de grupo?", "incompatibilidad horaria"),
    ("¿me convalidan lo que hice en otra carrera?", "reconocimiento"),
]


def evaluar(tabla, k=3):
    """Recall@k de las tres búsquedas, más el detalle pregunta a pregunta."""
    texto_de = {r["id"]: r["texto"] for r in tabla.to_arrow().to_pylist()}
    aciertos = {"densa": 0, "léxica": 0, "híbrida": 0}
    detalle = []

    for pregunta, esperado in CASOS:
        candidatos = {
            "densa": [r["id"] for r in buscar_densa(tabla, pregunta, k)],
            "léxica": [r["id"] for r in buscar_lexica(tabla, pregunta, k)],
            "híbrida": buscar_hibrida(tabla, pregunta, k),
        }
        fila = {}
        for nombre, ids in candidatos.items():
            ok = any(esperado.lower() in texto_de[i].lower() for i in ids)
            aciertos[nombre] += ok
            fila[nombre] = ok
        detalle.append((pregunta, fila))

    return aciertos, detalle


aciertos, detalle = evaluar(tabla)

print(f"Sobre {len(CASOS)} preguntas, troceado por sección:\n")
for nombre, n in aciertos.items():
    print(f"  recall@3 {nombre:8s} {n:2d}/{len(CASOS)}   {n / len(CASOS):.0%}")

Ahora el detalle, que es donde está lo interesante. `D` significa que la vectorial acertó, `L` que acertó la léxica y `H` que acertó la híbrida.

In [ ]:
print("D L H   pregunta")
print("-" * 60)
for pregunta, fila in detalle:
    marcas = "".join(
        letra if fila[clave] else "·"
        for letra, clave in [("D", "densa"), ("L", "léxica"), ("H", "híbrida")]
    )
    print(f"{' '.join(marcas)}   {pregunta}")

Leed la columna de la izquierda de arriba abajo. El patrón es el de la tabla de antes, ahora medido en lugar de afirmado:

* Las preguntas con **términos exactos** (el artículo 21, el TFG) las gana la léxica y las falla la vectorial.
* Las preguntas con **paráfrasis** (nota por acta, convalidar por reconocimiento) las gana la vectorial y las falla la léxica.
* **La híbrida no pierde ninguna de las dos.** Recupera todo lo que recuperaba cualquiera de ellas.

Y quedan un par de preguntas que fallan las tres. Eso también es información: significa que el sistema tiene un techo, y que ese techo no se sube tocando la búsqueda.

Merece la pena insistir en el orden de magnitud. Pasar de una búsqueda a la híbrida sube el recall unos veinte puntos, y esa mejora sale de cinco líneas de código y cero euros. Ninguna otra decisión de este cuaderno da tanto por tan poco.

### ¿Y el troceado?

Nos quedaba la pregunta del principio: ¿es mejor trocear por estructura que por tamaño fijo? Ahora se puede contestar con un número en lugar de con una intuición.

In [ ]:
tabla_fija = construir_tabla(db, "fijo", trocear_fijo)

for nombre, t in [("por sección", tabla), ("tamaño fijo", tabla_fija)]:
    aciertos, _ = evaluar(t)
    resumen = "  ".join(f"{m} {n}/{len(CASOS)}" for m, n in aciertos.items())
    print(f"{nombre:12s} ({t.count_rows():3d} trozos)   {resumen}")

El resultado incomoda un poco, y por eso conviene enseñarlo: **el troceado por estructura no gana**. La búsqueda híbrida da lo mismo con los dos, y en la léxica el troceado fijo incluso puede salir mejor, porque sus trozos son más uniformes y eso favorece a los algoritmos que pesan por longitud.

La lección no es que trocear dé igual. Es que **la ganancia del troceado es mucho menor que la de la búsqueda híbrida**, y en un proyecto real conviene gastar el esfuerzo en el orden en que lo dan los números y no en el orden en que lo dan las publicaciones de LinkedIn.

## Generar

Ya tenemos el trozo bueno. Falta la parte que le da nombre a esto: metérselo al modelo.

Usamos `Qwen3-0.6B`, que es diminuto. Sus respuestas van a ser flojas, y eso es deliberado: si el contraste entre responder con contexto y sin él se ve con un modelo de 600 millones de parámetros, con uno grande se ve mejor.

Una nota sobre `enable_thinking=False`: los Qwen3 razonan antes de responder por defecto, lo que aquí solo añadiría tokens y latencia sin mejorar nada. Es la decisión de la que habla el capítulo de [inferencia](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/inferencia.html) al hablar de razonar gastando más.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(ctx.modelo)
generador = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)

SISTEMA = (
    "Eres el asistente de la secretaría académica. Responde en una frase, "
    "citando la fecha o el artículo concreto. Si el contexto no contiene la "
    "respuesta, dilo en lugar de suponerla."
)


def responder(pregunta, contexto=None, max_tokens=100):
    usuario = pregunta if contexto is None else f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": usuario}]

    texto = tokenizador.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    entrada = tokenizador(texto, return_tensors="pt")

    with torch.no_grad():
        salida = generador.generate(
            **entrada,
            max_new_tokens=max_tokens,
            do_sample=False,  # temperatura 0: queremos poder comparar
            pad_token_id=tokenizador.eos_token_id,
        )

    nuevos = salida[0][entrada.input_ids.shape[1]:]
    return tokenizador.decode(nuevos, skip_special_tokens=True).strip()

In [ ]:
def responder_con_rag(pregunta, k=3):
    """El sistema completo: recuperar, montar el contexto y generar."""
    texto_de = {r["id"]: r["texto"] for r in tabla.to_arrow().to_pylist()}
    ids = buscar_hibrida(tabla, pregunta, k)
    contexto = "\n\n---\n\n".join(texto_de[i] for i in ids)
    return responder(pregunta, contexto), ids


pregunta = "¿Hasta cuándo puedo pedir la beca general?"

print("SIN CONTEXTO")
print(" ", responder(pregunta))

respuesta, ids = responder_con_rag(pregunta)
print("\nCON CONTEXTO RECUPERADO")
print(" ", respuesta)
print("\n  fuentes:", ids)

Ahí está el asunto entero de este capítulo en dos líneas de salida.

Sin contexto, el modelo responde con aplomo y se equivoca. No dice "no lo sé", que sería lo correcto: se inventa un plazo plausible. Con el contexto recuperado da la fecha buena y se puede comprobar de dónde la ha sacado.

Y esa última parte, la de poder comprobarlo, no es un detalle menor. La [normativa que hemos indexado](https://iraitzm.github.io/manual-ia-generativa/parts/normativa/leyes.html) dice en su artículo 22 que la información de un canal automatizado no sustituye a la resolución administrativa. Un sistema que cita el artículo del que sale su respuesta permite al alumno ir a comprobarlo. Uno que no lo cita, no.

In [ ]:
# Probad con varias. Fijaos en cuáles falla y contrastad con la tabla de recall:
# cuando el sistema responde mal, casi siempre es porque no recuperó bien.
for p in ["¿puedo defender el TFG si me queda una asignatura?",
          "¿cuántas convocatorias tengo por asignatura?",
          "¿me van a dar la beca?"]:
    respuesta, ids = responder_con_rag(p)
    print(f"P: {p}")
    print(f"R: {respuesta}")
    print(f"   fuentes: {ids}\n")

La última pregunta merece atención aparte. "¿Me van a dar la beca?" no tiene respuesta en el corpus, y la normativa dice expresamente que la Secretaría no está facultada para anticiparla.

Lo correcto es que el sistema se niegue. Comprobad si lo hace. Si en lugar de negarse os da una respuesta, acabáis de encontrar el fallo más caro de esta clase de sistemas: **no es equivocarse, es equivocarse con seguridad sobre algo que tiene consecuencias**.

## Ejercicios

**1. Las dos preguntas que fallan.** En la tabla de detalle hay dos preguntas que fallan las tres búsquedas. Averiguad cuáles son, mirad el trozo que debería salir y averiguad por qué no sale. Pista: mirad el tamaño del trozo y las palabras que comparte con la pregunta.

**2. Limpiar el índice.** El frontmatter de los ficheros se está indexando como contenido y aparece entre los resultados. Quitadlo antes de trocear y volved a medir. Fijaos en si el recall sube tanto como esperabais: puede que descubráis que un trozo malo entre los tres primeros molesta menos de lo que parecía.

**3. Subir el recall sin tocar la búsqueda.** Probad a añadir el título del documento al principio de cada trozo antes de calcular el vector. Es una técnica muy recomendada. Medidla con `evaluar()` antes de creérosla.

**4. El número de resultados.** Todo el cuaderno usa `k=3`. Medid el recall con 1, 3, 5 y 10, y pensad qué pasa con el coste: cada trozo extra son tokens que se pagan en cada consulta. ¿Dónde dejaríais el corte?

**5. La constante de RRF.** Está puesta a 60 porque es lo que dice la convención. Probad con 1, con 10 y con 1000, y explicad qué le pasa a la fusión en cada extremo.

**6. El troceado que falta.** Los dos troceadores de este cuaderno son los dos extremos. Escribid uno intermedio que corte por secciones pero parta las que pasen de 600 caracteres, y medidlo.

**7. Un modelo de embeddings mejor.** `paraphrase-multilingual-MiniLM-L12-v2` es pequeño y rápido. Probad con uno mayor y ved cuánto sube la vectorial. Cronometrad también el tiempo de indexado: la decisión no es solo de calidad.

## Lo que os lleváis

* La búsqueda vectorial **no** es mejor que la de palabras clave. Es mejor en unas cosas y peor en otras, y las cosas en las que es peor (códigos, números, referencias) abundan en el ámbito empresarial.
* **Normalizad los vectores.** Si no, la métrica que use el índice cambia los resultados y os vais a volver locos buscando por qué.
* **La búsqueda híbrida es la mejor relación entre esfuerzo y resultado** de todo el cuaderno: cinco líneas de RRF, veinte puntos de recall.
* El troceado importa menos de lo que se dice.
* **Sin medir no hay ingeniería.** Diez preguntas escritas en media hora os dicen más sobre vuestro sistema que cualquier cantidad de pruebas a ojo.

El siguiente paso es dejar de decidir nosotros cuándo hay que buscar y que lo decida el modelo. Eso es un agente, y es la [parte siguiente del manual](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/queesunagente.html).